# ⚙️ Advanced Trino-Iceberg Features

This notebook explores four advanced Iceberg features available through Trino + Polaris:

| # | Feature | What it shows |
|---|---------|---------------|
| 1 | **Object Store Layout** | Hashed (non-hierarchical) S3 path layout to avoid hot-spot partitions |
| 2 | **Table Statistics & ANALYZE** | Collecting column-level stats (`min`, `max`, `null_count`) for the query planner |
| 3 | **`add_files` Procedure** | Registering an existing Parquet file into an Iceberg table without rewriting data |
| 4 | **Metadata Caching** | Session-level flag to enable the metadata cache and how to verify it in the Trino UI |

> **Prerequisite:** Run `setup.ipynb` first so that the `lakehouse` catalog and Medallion namespaces exist.

---
## ⚙️ Step 0 — Configuration & Connections

In [ ]:
import boto3
import requests
import json
import time
import io
import random

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from botocore.client import Config
from trino.dbapi import connect

# --- S3 (SeaweedFS) ---
S3_ENDPOINT    = "http://seaweedfs:8333"
S3_ACCESS_KEY  = "lakehouse-admin"
S3_SECRET_KEY  = "lakehouse-secret-key"
S3_REGION      = "us-east-1"
S3_BUCKET      = "lakehouse"

# --- Trino ---
TRINO_HOST = "trino"
TRINO_PORT = 8080

# --- boto3 S3 client ---
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
    config=Config(signature_version="s3v4"),
)

# --- Trino connection ---
conn   = connect(host=TRINO_HOST, port=TRINO_PORT, user="admin", catalog="iceberg", schema="bronze")
cursor = conn.cursor()

def run(sql, display=True, quiet=False):
    """Execute SQL and pretty-print results."""
    if not quiet:
        print(f"▶ {sql[:120].strip()}{'...' if len(sql) > 120 else ''}")
    cursor.execute(sql)
    try:
        rows    = cursor.fetchall()
        columns = [d[0] for d in cursor.description] if cursor.description else []
        if display and rows and columns:
            widths = [
                max(len(str(c)), max((len(str(r[i])) for r in rows), default=0))
                for i, c in enumerate(columns)
            ]
            print(" | ".join(c.ljust(w) for c, w in zip(columns, widths)))
            print("-+-".join("-" * w for w in widths))
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows, columns
    except Exception:
        return [], []

print("✅ Configuration loaded and Trino connection established")

---
# 🗂️ Feature 1 — Object Store Layout (Hashed Paths)

By default Iceberg organises data files under a predictable path hierarchy derived from the table name and partition values (e.g. `s3://bucket/ns/table/data/part-00000.parquet`).  
In very large tables this creates **S3 hot-spots** because many files share the same key prefix and therefore land on the same storage shard.

Setting `'object_store_layout_enabled' = 'true'` makes Iceberg inject a **cryptographic hash prefix** into every data-file path, spreading writes evenly across the object store:

```
# Standard path
s3://lakehouse/bronze/hashed_table/data/00000-0-abc.parquet

# Hashed path
s3://lakehouse/bronze/hashed_table/data/4f9a3b2c/00000-0-abc.parquet
                                        ^^^^^^^^ — random hex prefix
```

We will prove this by querying the hidden `$files` metadata table, which exposes the physical path of each data file.

In [ ]:
# 1-a  Drop and recreate so the demo is idempotent
run("DROP TABLE IF EXISTS iceberg.bronze.hashed_table", display=False, quiet=True)

run("""
CREATE TABLE iceberg.bronze.hashed_table (
    id   INTEGER,
    name VARCHAR
) WITH (
    format                       = 'PARQUET',
    object_store_layout_enabled  = true
)
""")
print("✅ Table 'hashed_table' created with object_store_layout_enabled = true")

In [ ]:
# 1-b  Insert 5 rows — each commit writes a new data file
run("""
INSERT INTO iceberg.bronze.hashed_table VALUES
    (1, 'alpha'),
    (2, 'beta'),
    (3, 'gamma'),
    (4, 'delta'),
    (5, 'epsilon')
""")
print("✅ 5 rows inserted")

In [ ]:
# 1-c  Inspect physical file paths via the $files metadata table
print("📂 Physical file paths (from $files metadata table):")
print("-" * 70)

rows, _ = run(
    'SELECT file_path FROM iceberg.bronze."hashed_table$files"',
    display=False, quiet=True
)

for (path,) in rows:
    print(f"  {path}")

print()
print("🔍 Observation: notice the random hex segment embedded in the path.")
print("   Standard paths look like: .../data/00000-0-xxxx.parquet")
print("   Hashed paths look like:   .../data/<hex>/<hex>/00000-0-xxxx.parquet")

---
# 📊 Feature 2 — Table Statistics & ANALYZE

Iceberg stores **column-level statistics** (row count, null count, min, max, NDV) inside each data file's footer and in the snapshot summary.  
Trino's query planner can use these statistics to:
- Choose better join orders
- Push down more predicates
- Skip entire files (data skipping)

The `ANALYZE` command triggers a full table scan to compute and persist these statistics into Iceberg table metadata.

In [ ]:
# 2-a  Create stats_table
run("DROP TABLE IF EXISTS iceberg.bronze.stats_table", display=False, quiet=True)

run("""
CREATE TABLE iceberg.bronze.stats_table (
    id  INTEGER,
    val DOUBLE
) WITH (format = 'PARQUET')
""")
print("✅ Table 'stats_table' created")

In [ ]:
# 2-b  Insert 1,000 rows of random data in a single VALUES statement (batched for speed)
#      We build the VALUES list in Python to avoid 1000 round-trips.
random.seed(42)
values = ",\n    ".join(
    f"({i}, {round(random.uniform(0.0, 1000.0), 4)})"
    for i in range(1, 1001)
)

run(f"INSERT INTO iceberg.bronze.stats_table VALUES\n    {values}", display=False)
print("✅ 1,000 rows inserted into stats_table")

In [ ]:
# 2-c  Run ANALYZE to compute and persist column statistics
print("⏳ Running ANALYZE — this scans the table and writes statistics to Iceberg metadata...")
run("ANALYZE iceberg.bronze.stats_table", display=False)
print("✅ ANALYZE complete")

In [ ]:
# 2-d  Show the collected statistics
print("📊 Column statistics (SHOW STATS FOR stats_table):")
print()
run("SHOW STATS FOR iceberg.bronze.stats_table")

### 📖 Reading the output

| Column | Meaning |
|--------|---------|
| `column_name` | Column or `NULL` for the table-level row-count row |
| `data_size` | Estimated bytes for that column |
| `distinct_values_count` | NDV (used for selectivity estimation) |
| `nulls_fractions` | Fraction of NULLs (0.0 after our clean insert) |
| `row_count` | Total rows in the table (appears in the summary row) |
| `low_value` / `high_value` | Min / max values — confirms the range of `val` spans ~0–1000 |

---
# 📥 Feature 3 — The `add_files` Procedure

The `system.add_files` stored procedure lets you **register an existing Parquet (or ORC/Avro) file** that is already sitting in S3 into an Iceberg table — **without copying or rewriting the data**.  
This is the canonical pattern for migrating raw files into a governed Iceberg table.

**Workflow:**
1. Write a Parquet file to SeaweedFS with `boto3` + `pyarrow`
2. Create an empty Iceberg table with a matching schema
3. Call `CALL iceberg.system.add_files(...)` to register the file
4. Verify that the row count is now > 0

In [ ]:
# 3-a  Write a raw Parquet file to SeaweedFS using PyArrow
RAW_PARQUET_KEY = "raw-landing/products/products_raw.parquet"

# Build sample data
df = pd.DataFrame({
    "product_id":   [1001, 1002, 1003, 1004, 1005],
    "product_name": ["Widget A", "Widget B", "Gadget X", "Gadget Y", "Doohickey Z"],
    "price":        [9.99,  19.99, 49.99, 99.99, 4.99],
    "in_stock":     [True,  True,  False, True,  True],
})

# Serialise to an in-memory Parquet buffer
buf = io.BytesIO()
table = pa.Table.from_pandas(df, preserve_index=False)
pq.write_table(table, buf)
buf.seek(0)

# Upload to SeaweedFS
s3.put_object(Bucket=S3_BUCKET, Key=RAW_PARQUET_KEY, Body=buf.read())
print(f"✅ Raw Parquet file uploaded to s3://{S3_BUCKET}/{RAW_PARQUET_KEY}")
print(f"   Rows: {len(df)}, Columns: {list(df.columns)}")

In [ ]:
# 3-b  Create an empty Iceberg table with a schema matching the Parquet file
run("DROP TABLE IF EXISTS iceberg.bronze.products", display=False, quiet=True)

run("""
CREATE TABLE iceberg.bronze.products (
    product_id   INTEGER,
    product_name VARCHAR,
    price        DOUBLE,
    in_stock     BOOLEAN
) WITH (format = 'PARQUET')
""")
print("✅ Empty Iceberg table 'products' created")

In [ ]:
# 3-c  Confirm table is empty before add_files
rows, _ = run(
    "SELECT COUNT(*) AS row_count FROM iceberg.bronze.products",
    display=False, quiet=True
)
print(f"📭 Row count before add_files: {rows[0][0]}")

In [ ]:
# 3-d  Call add_files to register the raw Parquet file
#
# Signature:
#   CALL iceberg.system.add_files(
#       table         => '<catalog>.<schema>.<table>',
#       location      => 's3://<bucket>/<prefix>/',   -- directory OR single file
#       format        => 'PARQUET'
#   )
#
# We point at the directory so Iceberg discovers the file automatically.

location = f"s3://{S3_BUCKET}/raw-landing/products/"

add_files_sql = f"""
CALL iceberg.system.add_files(
    table    => 'iceberg.bronze.products',
    location => '{location}',
    format   => 'PARQUET'
)
"""

print(f"⏳ Calling add_files for location: {location}")
run(add_files_sql, display=False)
print("✅ add_files procedure completed")

In [ ]:
# 3-e  Verify row count is now > 0
rows, _ = run(
    "SELECT COUNT(*) AS row_count FROM iceberg.bronze.products",
    display=False, quiet=True
)
count = rows[0][0]
print(f"📊 Row count after add_files: {count}")

assert count > 0, "❌ add_files did not register any rows!"
print("✅ Assertion passed — add_files successfully registered the raw Parquet file!")

In [ ]:
# 3-f  Query the table to confirm the data is readable
print("📋 Products table contents (served from the registered raw Parquet file):")
run("SELECT * FROM iceberg.bronze.products ORDER BY product_id")

---
# 🚀 Feature 4 — Metadata Caching (Session Level)

Every time Trino resolves a table reference it must fetch the latest Iceberg **metadata JSON** from S3. For workloads that execute many queries per second against the same tables, these repeated metadata round-trips become a bottleneck.

The session-level property `iceberg.metadata_cache_enabled = true` instructs Trino to cache metadata files in coordinator memory between queries.  
Subsequent queries on the same table will hit the in-process cache rather than going back to S3.

> ⚠️ **Note:** The metadata cache trades freshness for speed. Use it during analytical sessions where table structure does not change frequently. Avoid it if you need to see uncommitted snapshot changes immediately.

In [ ]:
# 4-a  Enable the metadata cache for this session
run("SET SESSION iceberg.metadata_cache_enabled = true", display=False, quiet=True)
print("✅ Session property set: iceberg.metadata_cache_enabled = true")

In [ ]:
# 4-b  Run a representative analytical query (benefits from cached metadata)
print("⏳ Running an analytical query with metadata caching enabled...")
t0 = time.monotonic()

run("""
SELECT
    COUNT(*)          AS total_rows,
    MIN(val)          AS min_val,
    MAX(val)          AS max_val,
    AVG(val)          AS avg_val,
    STDDEV(val)       AS stddev_val
FROM iceberg.bronze.stats_table
""", quiet=True)

elapsed = time.monotonic() - t0
print(f"   First query elapsed: {elapsed:.2f}s  (metadata fetched from S3)")

# Run again — this time metadata should be served from the in-process cache
t1 = time.monotonic()
run("""
SELECT
    COUNT(*)          AS total_rows,
    MIN(val)          AS min_val,
    MAX(val)          AS max_val,
    AVG(val)          AS avg_val,
    STDDEV(val)       AS stddev_val
FROM iceberg.bronze.stats_table
""", display=False, quiet=True)
elapsed2 = time.monotonic() - t1
print(f"   Second query elapsed: {elapsed2:.2f}s  (metadata served from cache)")
print()
print(f"   Δ = {(elapsed - elapsed2):.2f}s  ({'faster' if elapsed2 < elapsed else 'similar'} on second run)")

### 🖥️ How to verify cache hit rate in the Trino UI

1. **Open the Trino Web UI** at [http://localhost:8443](http://localhost:8443)
2. Click on a recently completed query to open the **Query Details** page.
3. Navigate to the **"Stages"** or **"Operator Stats"** tab.
4. Look for the operator named **`IcebergMetadataCacheOperator`** or filter the stats table for the keyword `metadata`.
5. The column **`physicalInputDataSize`** will be `0 B` for any operator stage that was served entirely from the metadata cache — meaning no bytes were read from S3 for the metadata.

Alternatively, compare two consecutive identical queries:

| Metric | Query 1 (cold) | Query 2 (warm) |
|--------|---------------|---------------|
| Planning time | Higher | Lower |
| S3 requests (visible in SeaweedFS logs) | Several | Reduced |
| Total wall clock time | Baseline | Noticeably shorter for metadata-heavy workloads |

> **Tip:** Pair metadata caching with **ANALYZE** (Feature 2) — once stats are collected, the planner can skip entire data files *and* avoid repeated metadata reads, giving you compounding improvements.

In [ ]:
# 4-c  (Optional) Disable the cache at the end of the session
run("SET SESSION iceberg.metadata_cache_enabled = false", display=False, quiet=True)
print("✅ Session property reset: iceberg.metadata_cache_enabled = false")

---
## 🧹 Cleanup (Optional)

Uncomment and run this cell to drop all tables and the raw Parquet file created by this notebook.

In [ ]:
# # --- Drop Iceberg tables ---
# for tbl in ["iceberg.bronze.hashed_table", "iceberg.bronze.stats_table", "iceberg.bronze.products"]:
#     run(f"DROP TABLE IF EXISTS {tbl}", display=False, quiet=True)
#     print(f"🗑️  Dropped {tbl}")
#
# # --- Delete the raw Parquet file from SeaweedFS ---
# s3.delete_object(Bucket=S3_BUCKET, Key="raw-landing/products/products_raw.parquet")
# print("🗑️  Deleted s3://lakehouse/raw-landing/products/products_raw.parquet")